In [ ]:
import load_config
cfg = load_config.load_config()

import os
import ipywidgets as widgets
from IPython.display import display

bag_paths = list(cfg.keys())
list_widget = []
bag_paths_selected = []

for bag_path in bag_paths:
    if not os.path.exists(bag_path):
        # print(f"File not found: {bag_path}")
        None
    else:
        list_widget.append(
            widgets.Checkbox(value=False, description=bag_path,layout=widgets.Layout(width='100%')))
        

container = widgets.VBox(list_widget)

display(container)

btn = widgets.Button(description="Load bag(s)")
out = widgets.Output()
display(btn, out)

def on_click(b):
    with out:
        out.clear_output()
        selected = [cb.description for cb in list_widget if cb.value]
        print("Selected:", selected)
        global bag_paths_selected
        bag_paths_selected = selected

btn.on_click(on_click)

In [ ]:
import load_data
import load_config

required_topics = ["/estimated_robot_state", "odom", "joint_state", "/unitree/contact_state", "/unitree/imu", "/unitree/raw_contact"]

load_data.ros1_check_file(bag_paths_selected,required_topics)

In [ ]:
#----------------------------------HYSTERISIS PLOTTING AND THRESHOLDING--------------------------------------
bag_file = bag_paths_selected[0]

cfg = load_config.load_config()

if bag_file in cfg:
    if "threshold" in cfg[bag_file]:
        default_thresholds = cfg[bag_file]["threshold"]
else:
    default_thresholds = {
        'raw_contacts_0': 300 - 100,  # Adjust based on analysis
        'raw_contacts_1': 500 - 150,  # Adjust based on analysis
        'raw_contacts_2': 50 - 150,  # Adjust based on analysis
        'raw_contacts_3': 450 - 100,  # Adjust based on analysis (higher due to sensitivity)
    }
    
    

import bagpy
from bagpy import bagreader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

df_raw = load_data.ros1_load_topic(bag_file, "/unitree/raw_contact")

# Contact columns
contact_cols = ['raw_contacts_0', 'raw_contacts_1', 'raw_contacts_2', 'raw_contacts_3']
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

# Function to detect optimal threshold using multiple methods
def analyze_contact_thresholds(data, col_name):
    """Analyze data to suggest optimal contact detection threshold
       So we do a thorough analysing of the data here"""

    # Method 1: Statistical analysis
    mean_val = np.mean(data)
    std_val = np.std(data)
    median_val = np.median(data)

    # Method 2: Histogram-based (find valley between peaks if bimodal)
    hist, bin_edges = np.histogram(data, bins=50)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Method 3: Otsu's method (adapted for threshold selection)
    def otsu_threshold(data):
        hist, bin_edges = np.histogram(data, bins=256)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Normalize histogram
        hist = hist.astype(float) / hist.sum()

        max_variance = 0
        best_threshold = 0

        for i in range(1, len(hist)-1):
            # Background and foreground weights
            w0 = hist[:i].sum()
            w1 = hist[i:].sum()

            if w0 == 0 or w1 == 0:
                continue

            # Background and foreground means
            mu0 = (hist[:i] * bin_centers[:i]).sum() / w0 if w0 > 0 else 0
            mu1 = (hist[i:] * bin_centers[i:]).sum() / w1 if w1 > 0 else 0

            # Between-class variance
            variance = w0 * w1 * (mu0 - mu1)**2

            if variance > max_variance:
                max_variance = variance
                best_threshold = bin_centers[i]

        return best_threshold

    otsu_thresh = otsu_threshold(data)

    print(f"\n=== Threshold Analysis for {col_name} ===")
    print(f"Mean: {mean_val:.1f}")
    print(f"Median: {median_val:.1f}")
    print(f"Std: {std_val:.1f}")
    print(f"Mean + 0.5*Std: {mean_val + 0.5*std_val:.1f}")
    print(f"Mean + 1*Std: {mean_val + std_val:.1f}")
    print(f"Otsu threshold: {otsu_thresh:.1f}")
    print(f"Data range: {np.min(data):.1f} to {np.max(data):.1f}")

    return {
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'mean_plus_half_std': mean_val + 0.5*std_val,
        'mean_plus_std': mean_val + std_val,
        'otsu': otsu_thresh
    }

# Analyze each sensor
threshold_analysis = {}
for col in contact_cols:
    threshold_analysis[col] = analyze_contact_thresholds(df_raw[col], col)

# Manual threshold setting based on analysis
# You can adjust these based on the analysis output



# Function to apply threshold-based contact detection
def detect_contacts_threshold(data, threshold, hysteresis_factor=0.9, min_contact_duration=0.01):
    """
    Apply threshold-based contact detection with hysteresis and minimum duration filtering

    Parameters:
    - data: raw sensor data
    - threshold: contact detection threshold
    - hysteresis_factor: factor for lower threshold (prevents chattering)
    - Common “rule of thumb”  hysterisis of 5–15% below the main threshold usually works well for sensors that are moderately noisy.
    - min_contact_duration: minimum contact duration in seconds (1000Hz sampling)
    """

    # Hysteresis thresholds
    upper_thresh = threshold
    lower_thresh = threshold * hysteresis_factor

    # Initialize contact state
    contacts = np.zeros(len(data), dtype=bool)
    current_state = False

    # Apply hysteresis
    for i in range(len(data)):
        if not current_state and data[i] > upper_thresh:
            current_state = True
        elif current_state and data[i] < lower_thresh:
            current_state = False
        contacts[i] = current_state

    # Apply minimum duration filtering (remove very short contacts/gaps)
    # At500Hz sampling rate
    min_samples = int(min_contact_duration * 500)

    # Remove short contact periods
    contact_diff = np.diff(contacts.astype(int))
    contact_starts = np.where(contact_diff == 1)[0] + 1
    contact_ends = np.where(contact_diff == -1)[0] + 1

    # Handle edge cases
    if contacts[0]:
        contact_starts = np.insert(contact_starts, 0, 0)
    if contacts[-1]:
        contact_ends = np.append(contact_ends, len(contacts))


    # Filter short contacts
    ls,le = None,None
    for start, end in zip(contact_starts, contact_ends):

        if ls and le:
            if start - le < min_samples:
                contacts[le:start] = True
            pass

        if end - start < min_samples:
            ls,le = start,end
            #contacts[start:end] = False

    return contacts

# Apply threshold-based detection
print("Applying threshold-based contact detection...")
detected_contacts = {}

for col in contact_cols:
    # Use suggested threshold or manual override
    threshold = default_thresholds[col]

    # Detect contacts
    detected_contacts[col] = detect_contacts_threshold(
        df_raw[col],
        threshold=threshold,
        hysteresis_factor=0.9,
        min_contact_duration=0.05  # 50ms minimum contact
    )

    # Calculate contact statistics
    contact_percentage = np.mean(detected_contacts[col]) * 100
    print(f"{col}: Threshold={threshold:.1f}, Contact={contact_percentage:.1f}% of time")

# Create visualization comparing original vs threshold-based detection
fig, axes = plt.subplots(4, 1, figsize=(16*2, 12*2), sharex=True)

for i, col in enumerate(contact_cols):

    # Plot raw signal
    axes[i].plot(df_raw["Time"], df_raw[col], color=colors[i], alpha=0.7, linewidth=1, label='Raw Signal')

    # Plot threshold line
    axes[i].axhline(y=default_thresholds[col], color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    axes[i].axhline(y=default_thresholds[col]*0.9, color='red', linestyle='--', alpha=0.8, label=f'Threshold ({default_thresholds[col]})')

    # Create twin axis for contact states
    ax2 = axes[i].twinx()


    # Plot threshold-based detection
    #ax2.plot(df_raw["Time"], detected_contacts[col].astype(int),
    #        color='darkgreen', linewidth=2, alpha=0.8)

    ax2.fill_between(df_raw["Time"], 0, detected_contacts[col].astype(int),
                    alpha=0.2, color='green', label='Threshold-Based Contact')

    ax2.set_ylim(-0.1, 1.3)
    ax2.set_ylabel('Contact State', fontsize=10)
    ax2.tick_params(axis='y', labelsize=8)

    # Format main axis
    axes[i].set_ylabel("Raw Value [a.u.]", fontsize=10)
    axes[i].set_title(f"{col} - Threshold-Based Contact Detection", fontsize=12)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='y', labelsize=8)

    # Combine legends
    lines1, labels1 = axes[i].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[i].legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

axes[-1].set_xlabel("Time [s]", fontsize=12)
plt.tight_layout()
plt.show()

# Save the improved contact states
contact_df = df_raw[['Time'] + contact_cols].copy()
for col in contact_cols:
    contact_df[f'{col}_contact_threshold'] = detected_contacts[col].astype(int)

print(f"\nContact detection complete!")
print(f"New dataframe shape: {contact_df.shape}")
print(f"Columns: {contact_df.columns.tolist()}")

# Export for ML training
contact_df.to_csv('concrete_improved_contact_labels.csv', index=False)
print("Saved improved contact labels to 'concrete_improved_contact_labels.csv'")

    # Display sample of the results
print("\nSample of improved contact labels:")
print(contact_df[['Time', 'raw_contacts_0', 'raw_contacts_0_contact_threshold', \
                    'raw_contacts_3', 'raw_contacts_3_contact_threshold']].head(10))

In [ ]:
import pandas as pd
def merge_contact_labels(cleaned_csv_path, thresholded_csv_path, output_csv_path, contact_cols=None):
    if contact_cols is None:
        contact_cols = ['raw_contacts_0_contact_threshold', 'raw_contacts_1_contact_threshold', 
                        'raw_contacts_2_contact_threshold', 'raw_contacts_3_contact_threshold']
    try:
        # Load files inpputs
        cleaned_df = pd.read_csv(cleaned_csv_path)
        thresholded_df = pd.read_csv(thresholded_csv_path)
        
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return None

    print("Length cleaned:", len(cleaned_df))
    print("Length thresholded:", len(thresholded_df))

    # Ensure Time columns are sorted (REQUIRED for merge_asof)
    cleaned_df = cleaned_df.sort_values("Time").reset_index(drop=True)
    thresholded_df = thresholded_df.sort_values("Time").reset_index(drop=True)

    # ---- NEAREST-TIMESTAMP MERGE ----
    merged = pd.merge_asof(
        cleaned_df,
        thresholded_df[["Time"] + [f"{col}_contact_threshold" for col in contact_cols]],
        on="Time",
        direction="nearest",
        tolerance=0.002,      # 2 milliseconds tolerance
    )

    # Rebuild EE-contact list column
    merged["ee_contact"]= merged[contact_cols].astype(int).values.tolist()

    print("Length merged after nearest:", len(merged))

    # ---- REBUILD ee_contact LIST-COLUMN ----
    # merged["ee_contact"] = merged[
    #     [
    #         "raw_contacts_0_contact_threshold",
    #         "raw_contacts_1_contact_threshold",
    #         "raw_contacts_2_contact_threshold",
    #         "raw_contacts_3_contact_threshold",
    #     ]
    # ].astype(int).values.tolist()

    # ---- SAVE ----
    #output_path = "./data/dataset_path"
    merged.to_csv(output_csv_path, index=False)

    print("Saved:", output_csv_path)

    # ===== LOAD MERGED CONTACT LABELS (REFINED CONTACTS) =====
    contact_df = pd.read_csv(out)

    print("\nLoaded refined contact labels from:", output_csv_path)
    print("First 5 refined contact rows:")
    print(contact_df[[contact_cols]].head())

In [ ]:
import os
import re
import ast
import numpy as np
import pandas as pd

# -------------------------
# Helper Functions
# -------------------------
def fix_bracket_list_str(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    if s.startswith('[') and s.endswith(']'):
        content = s[1:-1].strip()
        parts = re.split(r'\s+', content)
        return "[" + ",".join(parts) + "]"
    return s

def safe_literal_eval(val):
    if pd.isna(val):
        return None
    try:
        val_fixed = fix_bracket_list_str(val)
        return ast.literal_eval(val_fixed)
    except Exception:
        try:
            nums = re.findall(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+', str(val))
            return [float(x) for x in nums]
        except Exception:
            return None

def to_fixed_length_array(x, length=12):
    if x is None:
        return np.full(length, np.nan, dtype=float)
    if isinstance(x, np.ndarray):
        arr = x
    else:
        try:
            arr = np.array(x, dtype=float)
        except Exception:
            return np.full(length, np.nan, dtype=float)
    if arr.size == length:
        return arr.astype(float)
    if arr.size < length:
        padded = np.full(length, np.nan, dtype=float)
        padded[:arr.size] = arr
        return padded
    return arr[:length].astype(float)

def quat_xyzw_to_rotmat(x, y, z, w):
    xx = x * x
    yy = y * y
    zz = z * z
    ww = w * w
    xy = x * y
    xz = x * z
    xw = x * w
    yz = y * z
    yw = y * w
    zw = z * w
    R = np.array([
        [1 - 2*(yy + zz), 2*(xy - w*z),     2*(xz + w*y)],
        [2*(xy + w*z),    1 - 2*(xx + zz),  2*(yz - w*x)],
        [2*(xz - w*y),    2*(yz + w*x),     1 - 2*(xx + yy)]
    ])
    return R

# -------------------------
# Core Processing Functions
# -------------------------
def load_refined_contacts(output_csv_path, contact_cols):
    """Load and prepare refined contact labels."""
    print("Loading refined contact labels from:", output_csv_path)
    refined_df = pd.read_csv(output_csv_path)
    if "Time" not in refined_df.columns:
        raise RuntimeError("Refined contact CSV must contain a 'Time' column for alignment.")
    refined_df["Time"] = refined_df["Time"].astype(float)
    refined_df = refined_df.sort_values("Time").reset_index(drop=True)
    print("First 5 refined contact rows:")
    print(refined_df[["Time"] + contact_cols].head())
    return refined_df

def parse_arrays_from_df(state_df, imu_df):
    """Parse EE positions, velocities, forces, joints, and IMU from DataFrames."""
    use_pos_col = "ee_pos_cleaned" if "ee_pos_cleaned" in state_df.columns else "ee_pos"
    use_vel_col = "ee_vel_cleaned" if "ee_vel_cleaned" in state_df.columns else "ee_vel"
    use_forces_col = "ee_forces_cleaned" if "ee_forces_cleaned" in state_df.columns else "ee_forces"

    parsed_pos = state_df[use_pos_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
    ee_pos_array = np.vstack(parsed_pos) if parsed_pos else np.empty((0,12))

    parsed_vel = state_df[use_vel_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
    ee_vel_array = np.vstack(parsed_vel) if parsed_vel else np.empty((0,12))

    parsed_forces = state_df[use_forces_col].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list()
    ee_forces_array = np.vstack(parsed_forces) if parsed_forces else np.empty((0,12))

    acc = imu_df[["linear_acceleration.x", "linear_acceleration.y", "linear_acceleration.z"]].to_numpy() if not imu_df.empty else np.empty((0, 3))
    omega = imu_df[["angular_velocity.x", "angular_velocity.y", "angular_velocity.z"]].to_numpy() if not imu_df.empty else np.empty((0, 3))

    try:
        q = np.array(state_df["joint_state.position"].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list())
        qd = np.array(state_df["joint_state.velocity"].apply(safe_literal_eval).apply(lambda x: to_fixed_length_array(x, 12)).to_list())
    except Exception as e:
        print("Error parsing joint states:", e)
        q = qd = np.empty((0, 12))

    return ee_pos_array, ee_vel_array, ee_forces_array, q, qd, acc, omega

def trim_arrays_to_min_length(aligned_contacts, q, qd, ee_pos_array, ee_vel_array, ee_forces_array, acc, omega):
    """Trim all arrays to the minimum length."""
    arrays_for_length = [aligned_contacts, q, qd, ee_pos_array, ee_vel_array]
    if acc.size > 0: arrays_for_length.append(acc)
    if omega.size > 0: arrays_for_length.append(omega)
    min_length = min([arr.shape[0] for arr in arrays_for_length]) if arrays_for_length else 0
    print(f"Trimming all to min_length = {min_length}")

    return (
        aligned_contacts[:min_length],
        q[:min_length],
        qd[:min_length],
        ee_pos_array[:min_length],
        ee_vel_array[:min_length],
        ee_forces_array[:min_length],
        acc[:min_length] if acc.size > 0 else acc,
        omega[:min_length] if omega.size > 0 else omega,
        min_length
    )

def convert_to_body_frame(ee_pos_array, ee_vel_array, state_df, min_length):
    """Convert EE positions and velocities from WORLD to BODY frame."""
    base_pos = state_df[["base_pose.position.x","base_pose.position.y","base_pose.position.z"]].to_numpy()[:min_length]
    qx = state_df["base_pose.orientation.x"].to_numpy()[:min_length]
    qy = state_df["base_pose.orientation.y"].to_numpy()[:min_length]
    qz = state_df["base_pose.orientation.z"].to_numpy()[:min_length]
    qw = state_df["base_pose.orientation.w"].to_numpy()[:min_length]

    ee_pos_body = np.zeros_like(ee_pos_array)
    ee_vel_body = np.zeros_like(ee_vel_array)
    for i in range(min_length):
        R = quat_xyzw_to_rotmat(qx[i], qy[i], qz[i], qw[i])
        R_bw = R.T
        for leg in range(4):
            idx = slice(leg*3, (leg+1)*3)
            pw = ee_pos_array[i, idx]
            vw = ee_vel_array[i, idx]
            pw_rel = pw - base_pos[i]
            pb = R_bw.dot(pw_rel)
            vb = R_bw.dot(vw)
            ee_pos_body[i, idx] = pb
            ee_vel_body[i, idx] = vb

    print("Converted EE kinematics to BODY frame.")
    return ee_pos_body, ee_vel_body

def process_contacts_and_reorder(aligned_contacts, q, qd, ee_pos_array, ee_vel_array):
    """Process contacts to decimal labels and reorder joints/EE arrays."""
    contacts_int = aligned_contacts.astype(int)
    reorder_contacts = contacts_int[:, [2, 0, 3, 1]]  # RF, LF, RH, LH
    decimal_contacts = np.dot(reorder_contacts, [8, 4, 2, 1])

    joint_order = [6,7,8,0,1,2,9,10,11,3,4,5]
    if q.shape[1] >= 12:
        q = q[:, joint_order]
        qd = qd[:, joint_order]
        if ee_pos_array.shape[1] >= 12:
            ee_pos_array = ee_pos_array[:, joint_order]
        if ee_vel_array.shape[1] >= 12:
            ee_vel_array = ee_vel_array[:, joint_order]

    return contacts_int, decimal_contacts, q, qd, ee_pos_array, ee_vel_array

def save_preprocessed_data(output_dir, base_name, q, qd, ee_pos_array, ee_vel_array, acc, omega, contacts_int, decimal_contacts):
    """Concatenate and save all preprocessed data as .npy files."""
    to_concat = [q, qd, ee_pos_array, ee_vel_array]
    if acc.size > 0: to_concat.append(acc)
    if omega.size > 0: to_concat.append(omega)
    preprocessed_data = np.concatenate(to_concat, axis=1)

    terrain_dir = os.path.join(output_dir, base_name)
    os.makedirs(terrain_dir, exist_ok=True)

    np.save(os.path.join(terrain_dir, "q.npy"), q)
    np.save(os.path.join(terrain_dir, "qd.npy"), qd)
    np.save(os.path.join(terrain_dir, "p.npy"), ee_pos_array)
    np.save(os.path.join(terrain_dir, "v.npy"), ee_vel_array)
    np.save(os.path.join(terrain_dir, "acc.npy"), acc)
    np.save(os.path.join(terrain_dir, "omega.npy"), omega)
    np.save(os.path.join(terrain_dir, "contacts.npy"), contacts_int)
    np.save(os.path.join(terrain_dir, "contacts_labels.npy"), decimal_contacts)
    np.save(os.path.join(terrain_dir, "preprocessed_data.npy"), preprocessed_data)

    print(f"Saved all data to {terrain_dir}")
    return terrain_dir

# -------------------------
# Main Processing Function (Updated)
# -------------------------
def prepare_ml_data(keyword, cleaned_csv_dir, output_csv_path, output_dir, contact_cols=None):
    if contact_cols is None:
        contact_cols = [
            "raw_contacts_0_contact_threshold",
            "raw_contacts_1_contact_threshold",
            "raw_contacts_2_contact_threshold",
            "raw_contacts_3_contact_threshold",
        ]
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load refined contacts
    refined_df = load_refined_contacts(output_csv_path, contact_cols)
    
    # Process each cleaned CSV
    cleaned_files = sorted([
        os.path.join(cleaned_csv_dir, f)
        for f in os.listdir(cleaned_csv_dir)
        if f.endswith("_cleaned.csv") and keyword in f
    ])
    print(f"\nFound {len(cleaned_files)} matching cleaned CSV files.")
    
    terrain_dirs = []
    errors = []
    
    for file in cleaned_files:
        try:
            base_name = os.path.basename(file).replace("_cleaned.csv", "")
            print(f"\nProcessing: {base_name}")

            # Load state and IMU
            state_df = pd.read_csv(file)
            imu_file = file.replace("_cleaned.csv", "_imu.csv")
            imu_df = pd.read_csv(imu_file) if os.path.exists(imu_file) else pd.DataFrame()

            if "Time" not in state_df.columns:
                raise RuntimeError(f"State CSV {file} does not contain 'Time' column.")
            state_df["Time"] = state_df["Time"].astype(float)
            state_df = state_df.sort_values("Time").reset_index(drop=True)

            # Align contacts
            merged = pd.merge_asof(
                state_df[["Time"]],
                refined_df[["Time"] + contact_cols],
                on="Time",
                direction="nearest",
                tolerance=0.01
            )
            mask_matched = merged[contact_cols].notnull().all(axis=1)
            if mask_matched.sum() == 0:
                print("No matched rows; skipping.")
                continue
            matched_state_df = state_df[mask_matched].reset_index(drop=True)
            aligned_contacts = merged.loc[mask_matched, contact_cols].to_numpy().astype(int)

            # Parse arrays
            ee_pos_array, ee_vel_array, ee_forces_array, q, qd, acc, omega = parse_arrays_from_df(matched_state_df, imu_df)

            # Trim
            aligned_contacts, q, qd, ee_pos_array, ee_vel_array, ee_forces_array, acc, omega, min_length = trim_arrays_to_min_length(
                aligned_contacts, q, qd, ee_pos_array, ee_vel_array, ee_forces_array, acc, omega
            )

            # Convert to body frame
            ee_pos_array, ee_vel_array = convert_to_body_frame(ee_pos_array, ee_vel_array, matched_state_df, min_length)

            # Process contacts and reorder
            contacts_int, decimal_contacts, q, qd, ee_pos_array, ee_vel_array = process_contacts_and_reorder(
                aligned_contacts, q, qd, ee_pos_array, ee_vel_array
            )

            # Save
            terrain_dir = save_preprocessed_data(output_dir, base_name, q, qd, ee_pos_array, ee_vel_array, acc, omega, contacts_int, decimal_contacts)
            terrain_dirs.append(terrain_dir)

        except Exception as e:
            print(f"Failed to process {file}: {e}")
            errors.append(file)
    
    return {'terrain_dirs': terrain_dirs, 'errors': errors}

In [ ]:
# ===========================================
# FULL PIPELINE: MERGE + PREPARE ML DATA
# ===========================================

# Step 1: Merge
merged_csv_path = "./data/dataset_path"
merged_df = merge_contact_labels(
    cleaned_csv_path="./data/dataset_path",
    thresholded_csv_path="concrete_improved_contact_labels.csv",
    output_csv_path=merged_csv_path
)

# Step 2: Prepare ML data (now includes all steps: parse, trim, convert, reorder, save)
if merged_df is not None:
    result = prepare_ml_data(
        keyword="legro_2025-06-09-14-23-57",
        cleaned_csv_dir="./data/dataset_path",
        refined_contact_path=merged_csv_path,
        output_dir="./data/dataset_path"
    )
    
    print("\nPipeline complete!")
    print("Processed terrain dirs:", result['terrain_dirs'])
    print("Errors:", result['errors'])
else:
    print("Merge failed; skipping ML preparation.")